# Financial Transaction Risk & Fraud Analytics — Data Profiling and Risk Scoring

This notebook follows the full workflow from data quality checks and feature engineering through rule-based scoring, Isolation Forest anomaly detection, final risk scoring, evaluation, and export.

**Important:** the notebook contains intermediate risk-score versions for comparison. Use the section labelled **FINAL EVALUATION — FRAUD AND FRAUD-VALUE CAPTURE** and the final export section for the corrected final score. Outputs have been cleared so results must be regenerated by running the notebook in order.

## 1. Data Loading

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root whether Jupyter starts in the project root or notebooks/.
cwd = Path.cwd()
if (cwd / "data" / "raw" / "paysim.csv").exists():
    project_root = cwd
elif (cwd.parent / "data" / "raw" / "paysim.csv").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find data/raw/paysim.csv. Place paysim.csv in the project's "
        "data/raw folder and launch Jupyter from the project root or notebooks folder."
    )

DATA_PATH = project_root / "data" / "raw" / "paysim.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded from: {DATA_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head()

Dataset loaded from: c:\Users\preet\OneDrive\Desktop\Financial Transaction Risk & Fraud Analytics Platform\data\raw\paysim.csv
Rows: 6,362,620
Columns: 11


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## 2. Data Quality Assessment

In [2]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [3]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
step,6362620.0,NaN,NaN,NaN,243.397246,142.331971,1.0,156.0,239.0,335.0,743.0
type,6362620,5,CASH_OUT,2237500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,6362620.0,NaN,NaN,NaN,179861.903549,603858.231463,0.0,13389.57,74871.94,208721.4775,92445516.64
nameOrig,6362620,6353307,C1677795071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceOrg,6362620.0,NaN,NaN,NaN,833883.104074,2888242.673038,0.0,0.0,14208.0,107315.175,59585040.37
newbalanceOrig,6362620.0,NaN,NaN,NaN,855113.668579,2924048.502954,0.0,0.0,0.0,144258.41,49585040.37
nameDest,6362620,2722362,C1286084959,113,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceDest,6362620.0,NaN,NaN,NaN,1100701.66652,3399180.112994,0.0,0.0,132705.665,943036.7075,356015889.35
newbalanceDest,6362620.0,NaN,NaN,NaN,1224996.398202,3674128.94212,0.0,0.0,214661.44,1111909.25,356179278.92
isFraud,6362620.0,NaN,NaN,NaN,0.001291,0.035905,0.0,0.0,0.0,0.0,1.0


## 3. Fraud Distribution

In [4]:
df.isnull().sum().sort_values(ascending=False)

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

## 4. Transaction Type Analysis

In [5]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


## 5. Financial Behavior Analysis

In [6]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [7]:
print(df["isFraud"].value_counts())

print("\nFraud rate:")
print(f"{df['isFraud'].mean() * 100:.4f}%")

isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud rate:
0.1291%


In [8]:
pd.crosstab(
    df["isFlaggedFraud"],
    df["isFraud"],
    margins=True
)

isFraud,0,1,All
isFlaggedFraud,,,
0,6354407,8197,6362604
1,0,16,16
All,6354407,8213,6362620


In [9]:
financial_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

df[financial_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
amount,6362620.0,1.798619e+05,6.038582e+05,0.0,13389.57,74871.940,2.087215e+05,9.244552e+07
oldbalanceOrg,6362620.0,8.338831e+05,2.888243e+06,0.0,0.00,14208.000,1.073152e+05,5.958504e+07
newbalanceOrig,6362620.0,8.551137e+05,2.924049e+06,0.0,0.00,0.000,1.442584e+05,4.958504e+07
oldbalanceDest,6362620.0,1.100702e+06,3.399180e+06,0.0,0.00,132705.665,9.430367e+05,3.560159e+08
newbalanceDest,6362620.0,1.224996e+06,3.674129e+06,0.0,0.00,214661.440,1.111909e+06,3.561793e+08


In [10]:
print("Negative amounts:", (df["amount"] < 0).sum())
print("Zero amounts:", (df["amount"] == 0).sum())

Negative amounts: 0
Zero amounts: 16


In [11]:

# Inspect zero-amount transactions
zero_amount = df[df["amount"] == 0]

print(f"Zero-amount transactions: {len(zero_amount)}")

zero_amount[
    [
        "step",
        "type",
        "amount",
        "nameOrig",
        "oldbalanceOrg",
        "newbalanceOrig",
        "nameDest",
        "oldbalanceDest",
        "newbalanceDest",
        "isFraud",
        "isFlaggedFraud"
    ]
]

Zero-amount transactions: 16


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
2736447,212,CASH_OUT,0.0,C1510987794,0.0,0.0,C1696624817,0.00,0.00,1,0
3247298,250,CASH_OUT,0.0,C521393327,0.0,0.0,C480398193,0.00,0.00,1,0
3760289,279,CASH_OUT,0.0,C539112012,0.0,0.0,C1106468520,538547.63,538547.63,1,0
5563714,387,CASH_OUT,0.0,C1294472700,0.0,0.0,C1325541393,7970766.57,7970766.57,1,0
5996408,425,CASH_OUT,0.0,C832555372,0.0,0.0,C1462759334,76759.90,76759.90,1,0
5996410,425,CASH_OUT,0.0,C69493310,0.0,0.0,C719711728,2921531.34,2921531.34,1,0
6168500,554,CASH_OUT,0.0,C10965156,0.0,0.0,C1493336195,230289.66,230289.66,1,0
6205440,586,CASH_OUT,0.0,C1303719003,0.0,0.0,C900608348,1328472.86,1328472.86,1,0
6266414,617,CASH_OUT,0.0,C1971175979,0.0,0.0,C1352345416,0.00,0.00,1,0
6281483,646,CASH_OUT,0.0,C2060908932,0.0,0.0,C1587892888,0.00,0.00,1,0


In [12]:
# Fraud by transaction type
fraud_by_type = (
    df.groupby("type")
      .agg(
          transactions=("isFraud", "size"),
          fraud_transactions=("isFraud", "sum"),
          transaction_value=("amount", "sum"),
          fraud_value=("amount", lambda x: x[df.loc[x.index, "isFraud"] == 1].sum())
      )
      .reset_index()
)

fraud_by_type["fraud_rate_pct"] = (
    fraud_by_type["fraud_transactions"]
    / fraud_by_type["transactions"]
    * 100
)

fraud_by_type.sort_values("fraud_rate_pct", ascending=False)

,type,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct
4,TRANSFER,532909,4097,4.852920e+11,6.067213e+09,0.768799
1,CASH_OUT,2237500,4116,3.944130e+11,5.989202e+09,0.183955
0,CASH_IN,1399284,0,2.363674e+11,0.000000e+00,0.000000
2,DEBIT,41432,0,2.271992e+08,0.000000e+00,0.000000
3,PAYMENT,2151495,0,2.809337e+10,0.000000e+00,0.000000


In [13]:
# Fraud vs normal transaction amounts
df.groupby("isFraud")["amount"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

,count,mean,median,std,min,max
isFraud,,,,,,
0,6354407,1.781970e+05,74684.72,5.962370e+05,0.01,92445516.64
1,8213,1.467967e+06,441423.44,2.404253e+06,0.00,10000000.00


In [14]:
df_analytics = df.copy()

# ---------------------------------------
# 1. Transaction ID
# ---------------------------------------
df_analytics["transaction_id"] = (
    "TXN_" + (df_analytics.index + 1).astype(str)
)

# ---------------------------------------
# 2. Time features
# ---------------------------------------
# PaySim 'step' represents an hourly time step
df_analytics["hour"] = df_analytics["step"] % 24
df_analytics["day"] = (df_analytics["step"] // 24) + 1

# ---------------------------------------
# 3. Balance changes
# ---------------------------------------
df_analytics["origin_balance_change"] = (
    df_analytics["oldbalanceOrg"]
    - df_analytics["newbalanceOrig"]
)

df_analytics["destination_balance_change"] = (
    df_analytics["newbalanceDest"]
    - df_analytics["oldbalanceDest"]
)

# ---------------------------------------
# 4. Balance reconciliation errors
# ---------------------------------------
df_analytics["origin_balance_error"] = (
    df_analytics["oldbalanceOrg"]
    - df_analytics["amount"]
    - df_analytics["newbalanceOrig"]
)

df_analytics["destination_balance_error"] = (
    df_analytics["oldbalanceDest"]
    + df_analytics["amount"]
    - df_analytics["newbalanceDest"]
)

# ---------------------------------------
# 5. Zero amount indicator
# ---------------------------------------
df_analytics["is_zero_amount"] = (
    df_analytics["amount"] == 0
).astype(int)

# ---------------------------------------
# 6. Amount relative to origin balance
# ---------------------------------------
df_analytics["amount_to_origin_balance"] = np.where(
    df_analytics["oldbalanceOrg"] > 0,
    df_analytics["amount"] / df_analytics["oldbalanceOrg"],
    0
)

# ---------------------------------------
# 7. Amount relative to destination balance
# ---------------------------------------
df_analytics["amount_to_destination_balance"] = np.where(
    df_analytics["oldbalanceDest"] > 0,
    df_analytics["amount"] / df_analytics["oldbalanceDest"],
    0
)

print(df_analytics.shape)

(6362620, 21)


In [15]:
df_analytics[
    [
        "transaction_id",
        "type",
        "amount",
        "hour",
        "day",
        "origin_balance_change",
        "destination_balance_change",
        "origin_balance_error",
        "destination_balance_error",
        "is_zero_amount",
        "amount_to_origin_balance",
        "amount_to_destination_balance"
    ]
].head(10)

,transaction_id,type,amount,hour,day,origin_balance_change,destination_balance_change,origin_balance_error,destination_balance_error,is_zero_amount,amount_to_origin_balance,amount_to_destination_balance
0,TXN_1,PAYMENT,9839.64,1,1,9839.64,0.00,0.000000e+00,9839.64,0,0.057834,0.000000
1,TXN_2,PAYMENT,1864.28,1,1,1864.28,0.00,0.000000e+00,1864.28,0,0.087735,0.000000
2,TXN_3,TRANSFER,181.00,1,1,181.00,0.00,0.000000e+00,181.00,0,1.000000,0.000000
3,TXN_4,CASH_OUT,181.00,1,1,181.00,-21182.00,0.000000e+00,21363.00,0,1.000000,0.008545
4,TXN_5,PAYMENT,11668.14,1,1,11668.14,0.00,0.000000e+00,11668.14,0,0.280795,0.000000
5,TXN_6,PAYMENT,7817.71,1,1,7817.71,0.00,0.000000e+00,7817.71,0,0.145149,0.000000
6,TXN_7,PAYMENT,7107.77,1,1,7107.77,0.00,0.000000e+00,7107.77,0,0.038799,0.000000
7,TXN_8,PAYMENT,7861.64,1,1,7861.64,0.00,0.000000e+00,7861.64,0,0.044646,0.000000
8,TXN_9,PAYMENT,4024.36,1,1,2671.00,0.00,-1.353360e+03,4024.36,0,1.506687,0.000000
9,TXN_10,DEBIT,5337.77,1,1,5337.77,-1549.21,-7.275958e-12,6886.98,0,0.127943,0.127399


In [16]:
df_analytics[
    [
        "origin_balance_error",
        "destination_balance_error",
        "amount_to_origin_balance",
        "amount_to_destination_balance"
    ]
].describe()

,origin_balance_error,destination_balance_error,amount_to_origin_balance,amount_to_destination_balance
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,-2.010925e+05,5.556717e+04,8.171346e+01,5.923620e+00
std,6.066505e+05,4.415288e+05,3.395705e+03,2.941265e+03
min,-9.244552e+07,-7.588573e+07,0.000000e+00,0.000000e+00
25%,-2.496411e+05,0.000000e+00,0.000000e+00,0.000000e+00
50%,-6.867726e+04,3.500490e+03,7.681118e-02,2.830899e-02
75%,-2.954230e+03,2.935305e+04,2.219448e+00,2.745522e-01
max,1.000000e-02,1.319123e+07,3.925476e+06,5.542639e+06


In [17]:
# ---------------------------------------
# Robust risk indicators
# ---------------------------------------

# 1. Transaction type risk
# TRANSFER and CASH_OUT are the fraud-prone transaction types
df_analytics["type_risk_flag"] = (
    df_analytics["type"].isin(["TRANSFER", "CASH_OUT"])
).astype(int)

# 2. Zero amount
df_analytics["zero_amount_flag"] = (
    df_analytics["amount"] == 0
).astype(int)

# 3. Origin balance depletion
# Transaction consumes most/all of the origin account balance
df_analytics["origin_depletion_ratio"] = np.where(
    df_analytics["oldbalanceOrg"] > 0,
    df_analytics["amount"] / df_analytics["oldbalanceOrg"],
    0
)

df_analytics["full_origin_depletion"] = (
    (df_analytics["oldbalanceOrg"] > 0) &
    (df_analytics["newbalanceOrig"] == 0) &
    (df_analytics["amount"] > 0)
).astype(int)

# 4. Destination balance change mismatch
df_analytics["destination_balance_mismatch"] = (
    np.abs(df_analytics["destination_balance_error"]) > 1
).astype(int)

# 5. Origin balance mismatch
df_analytics["origin_balance_mismatch"] = (
    np.abs(df_analytics["origin_balance_error"]) > 1
).astype(int)

# 6. Existing fraud flag
df_analytics["existing_flag"] = df_analytics["isFlaggedFraud"]

print("Risk features created.")

Risk features created.


In [18]:
risk_features = [
    "type_risk_flag",
    "zero_amount_flag",
    "full_origin_depletion",
    "destination_balance_mismatch",
    "origin_balance_mismatch",
    "existing_flag"
]

risk_summary = []

for feature in risk_features:
    summary = df_analytics.groupby(feature)["isFraud"].agg(
        transactions="count",
        fraud_transactions="sum"
    ).reset_index()

    summary["fraud_rate_pct"] = (
        summary["fraud_transactions"]
        / summary["transactions"]
        * 100
    )

    summary["feature"] = feature
    risk_summary.append(summary)

risk_summary = pd.concat(risk_summary, ignore_index=True)

risk_summary

,type_risk_flag,transactions,fraud_transactions,fraud_rate_pct,feature,zero_amount_flag,full_origin_depletion,destination_balance_mismatch,origin_balance_mismatch,existing_flag
0,0.0,3592211,0,0.000000,type_risk_flag,NaN,NaN,NaN,NaN,NaN
1,1.0,2770409,8213,0.296454,type_risk_flag,NaN,NaN,NaN,NaN,NaN
2,NaN,6362604,8197,0.128831,zero_amount_flag,0.0,NaN,NaN,NaN,NaN
3,NaN,16,16,100.000000,zero_amount_flag,1.0,NaN,NaN,NaN,NaN
4,NaN,4842039,201,0.004151,full_origin_depletion,NaN,0.0,NaN,NaN,NaN
5,NaN,1520581,8012,0.526904,full_origin_depletion,NaN,1.0,NaN,NaN,NaN
6,NaN,2538786,3977,0.156650,destination_balance_mismatch,NaN,NaN,0.0,NaN,NaN
7,NaN,3823834,4236,0.110779,destination_balance_mismatch,NaN,NaN,1.0,NaN,NaN
8,NaN,1361836,8168,0.599779,origin_balance_mismatch,NaN,NaN,NaN,0.0,NaN
9,NaN,5000784,45,0.000900,origin_balance_mismatch,NaN,NaN,NaN,1.0,NaN


In [19]:
# Calculate 95th and 99th percentile amount thresholds by transaction type

amount_thresholds = (
    df_analytics.groupby("type")["amount"]
    .quantile([0.95, 0.99])
    .unstack()
    .reset_index()
)

amount_thresholds.columns = [
    "type",
    "amount_p95",
    "amount_p99"
]

amount_thresholds

,type,amount_p95,amount_p99
0,CASH_IN,4.120051e+05,5.508709e+05
1,CASH_OUT,4.278772e+05,5.796541e+05
2,DEBIT,1.479535e+04,5.081798e+04
3,PAYMENT,3.783566e+04,5.950011e+04
4,TRANSFER,2.674587e+06,1.000000e+07


In [20]:
# Merge thresholds into analytical dataset
df_analytics = df_analytics.merge(
    amount_thresholds,
    on="type",
    how="left"
)

# Amount anomaly indicators
df_analytics["amount_above_p95"] = (
    df_analytics["amount"] > df_analytics["amount_p95"]
).astype(int)

df_analytics["amount_above_p99"] = (
    df_analytics["amount"] > df_analytics["amount_p99"]
).astype(int)

In [21]:
amount_risk_summary = (
    df_analytics
    .groupby(["type", "amount_above_p95"])
    ["isFraud"]
    .agg(
        transactions="count",
        fraud_transactions="sum"
    )
    .reset_index()
)

amount_risk_summary["fraud_rate_pct"] = (
    amount_risk_summary["fraud_transactions"]
    / amount_risk_summary["transactions"]
    * 100
)

amount_risk_summary

,type,amount_above_p95,transactions,fraud_transactions,fraud_rate_pct
0,CASH_IN,0,1329319,0,0.000000
1,CASH_IN,1,69965,0,0.000000
2,CASH_OUT,0,2125625,2043,0.096113
3,CASH_OUT,1,111875,2073,1.852961
4,DEBIT,0,39360,0,0.000000
5,DEBIT,1,2072,0,0.000000
6,PAYMENT,0,2043920,0,0.000000
7,PAYMENT,1,107575,0,0.000000
8,TRANSFER,0,506263,3439,0.679291
9,TRANSFER,1,26646,658,2.469414


In [22]:
amount_99_summary = (
    df_analytics
    .groupby(["type", "amount_above_p99"])
    ["isFraud"]
    .agg(
        transactions="count",
        fraud_transactions="sum"
    )
    .reset_index()
)

amount_99_summary["fraud_rate_pct"] = (
    amount_99_summary["fraud_transactions"]
    / amount_99_summary["transactions"]
    * 100
)

amount_99_summary

,type,amount_above_p99,transactions,fraud_transactions,fraud_rate_pct
0,CASH_IN,0,1385291,0,0.000000
1,CASH_IN,1,13993,0,0.000000
2,CASH_OUT,0,2215125,2310,0.104283
3,CASH_OUT,1,22375,1806,8.071508
4,DEBIT,0,41017,0,0.000000
5,DEBIT,1,415,0,0.000000
6,PAYMENT,0,2129980,0,0.000000
7,PAYMENT,1,21515,0,0.000000
8,TRANSFER,0,530466,4097,0.772340
9,TRANSFER,1,2443,0,0.000000


In [23]:
# ============================================
# RISK SCORING ENGINE
# ============================================

df_analytics["risk_score_raw"] = (
    df_analytics["existing_flag"] * 30
    + df_analytics["zero_amount_flag"] * 30
    + df_analytics["origin_balance_mismatch"] * 20
    + df_analytics["full_origin_depletion"] * 15
    + df_analytics["type_risk_flag"] * 10
    + df_analytics["amount_above_p95"] * 10
    + df_analytics["destination_balance_mismatch"] * 5
)

# Cap score at 100
df_analytics["risk_score"] = (
    df_analytics["risk_score_raw"].clip(upper=100)
)

# Risk categories
df_analytics["risk_level"] = pd.cut(
    df_analytics["risk_score"],
    bins=[-1, 30, 60, 80, 100],
    labels=["Low", "Medium", "High", "Critical"]
)

df_analytics[
    [
        "transaction_id",
        "type",
        "amount",
        "risk_score",
        "risk_level",
        "isFraud"
    ]
].head(20)

,transaction_id,type,amount,risk_score,risk_level,isFraud
0,TXN_1,PAYMENT,9839.64,5,Low,0
1,TXN_2,PAYMENT,1864.28,5,Low,0
2,TXN_3,TRANSFER,181.00,30,Low,1
3,TXN_4,CASH_OUT,181.00,30,Low,1
4,TXN_5,PAYMENT,11668.14,5,Low,0
5,TXN_6,PAYMENT,7817.71,5,Low,0
6,TXN_7,PAYMENT,7107.77,5,Low,0
7,TXN_8,PAYMENT,7861.64,5,Low,0
8,TXN_9,PAYMENT,4024.36,40,Medium,0
9,TXN_10,DEBIT,5337.77,5,Low,0


In [24]:
risk_performance = (
    df_analytics
    .groupby("risk_level", observed=True)
    .agg(
        transactions=("transaction_id", "count"),
        fraud_transactions=("isFraud", "sum"),
        transaction_value=("amount", "sum"),
        fraud_value=("amount", lambda x:
                     x[df_analytics.loc[x.index, "isFraud"] == 1].sum())
    )
    .reset_index()
)

risk_performance["fraud_rate_pct"] = (
    risk_performance["fraud_transactions"]
    / risk_performance["transactions"]
    * 100
)

risk_performance

,risk_level,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct
0,Low,4555573,5595,5.291770e+11,3.682698e+09,0.122817
1,Medium,1807031,2602,6.151382e+11,8.295932e+09,0.143993
2,High,16,16,7.778556e+07,7.778556e+07,100.000000


In [25]:
risk_distribution = (
    df_analytics["risk_level"]
    .value_counts()
    .sort_index()
)

risk_distribution

risk_level
Low         4555573
Medium      1807031
High             16
Critical          0
Name: count, dtype: int64

In [26]:
fraud_by_risk = pd.crosstab(
    df_analytics["risk_level"],
    df_analytics["isFraud"],
    normalize="index"
) * 100

fraud_by_risk

isFraud,0,1
risk_level,,
Low,99.877183,0.122817
Medium,99.856007,0.143993
High,0.000000,100.000000


In [27]:
# ============================================
# RISK SCORING ENGINE — VERSION 2
# ============================================

df_analytics["risk_score_v2_raw"] = (
    df_analytics["existing_flag"] * 35
    + df_analytics["zero_amount_flag"] * 35
    + df_analytics["origin_balance_mismatch"] * 25
    + df_analytics["full_origin_depletion"] * 20
    + df_analytics["type_risk_flag"] * 10
    + df_analytics["amount_above_p95"] * 10
    + df_analytics["destination_balance_mismatch"] * 5
)

df_analytics["risk_score_v2"] = (
    df_analytics["risk_score_v2_raw"].clip(upper=100)
)

df_analytics["risk_level_v2"] = pd.cut(
    df_analytics["risk_score_v2"],
    bins=[-1, 19, 39, 69, 100],
    labels=["Low", "Medium", "High", "Critical"]
)

df_analytics[
    [
        "transaction_id",
        "type",
        "amount",
        "risk_score_v2",
        "risk_level_v2",
        "isFraud"
    ]
].head(20)

,transaction_id,type,amount,risk_score_v2,risk_level_v2,isFraud
0,TXN_1,PAYMENT,9839.64,5,Low,0
1,TXN_2,PAYMENT,1864.28,5,Low,0
2,TXN_3,TRANSFER,181.00,35,Medium,1
3,TXN_4,CASH_OUT,181.00,35,Medium,1
4,TXN_5,PAYMENT,11668.14,5,Low,0
5,TXN_6,PAYMENT,7817.71,5,Low,0
6,TXN_7,PAYMENT,7107.77,5,Low,0
7,TXN_8,PAYMENT,7861.64,5,Low,0
8,TXN_9,PAYMENT,4024.36,50,High,0
9,TXN_10,DEBIT,5337.77,5,Low,0


In [28]:
fraud_by_risk_v2 = pd.crosstab(
    df_analytics["risk_level_v2"],
    df_analytics["isFraud"],
    normalize="index"
) * 100

fraud_by_risk_v2


isFraud,0,1
risk_level_v2,,
Low,99.999852,0.000148
Medium,99.825419,0.174581
High,99.855146,0.144854
Critical,99.851163,0.148837


In [29]:
risk_performance_v2 = (
    df_analytics
    .groupby("risk_level_v2", observed=True)
    .agg(
        transactions=("transaction_id", "count"),
        fraud_transactions=("isFraud", "sum"),
        transaction_value=("amount", "sum"),
        fraud_value=("amount", lambda x:
            x[df_analytics.loc[x.index, "isFraud"] == 1].sum())
    )
    .reset_index()
)

risk_performance_v2["fraud_rate_pct"] = (
    risk_performance_v2["fraud_transactions"]
    / risk_performance_v2["transactions"]
    * 100
)

risk_performance_v2

,risk_level_v2,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct
0,Low,1351901,2,3.643636e+10,1.036516e+06,0.000148
1,Medium,3203672,5593,4.927406e+11,3.681662e+09,0.174581
2,High,1796297,2602,5.897154e+11,8.295932e+09,0.144854
3,Critical,10750,16,2.550060e+10,7.778556e+07,0.148837


In [30]:
high_risk_mask = df_analytics["risk_level_v2"].isin(
    ["High", "Critical"]
)

fraud_total = df_analytics["isFraud"].sum()

fraud_captured = df_analytics.loc[
    high_risk_mask, "isFraud"
].sum()

fraud_capture_rate = (
    fraud_captured / fraud_total * 100
)

print(f"Total fraud: {fraud_total:,}")
print(f"Fraud captured by High/Critical: {fraud_captured:,}")
print(f"Fraud capture rate: {fraud_capture_rate:.2f}%")

Total fraud: 8,213
Fraud captured by High/Critical: 2,618
Fraud capture rate: 31.88%


In [31]:
fraud_value_total = df_analytics.loc[
    df_analytics["isFraud"] == 1,
    "amount"
].sum()

fraud_value_captured = df_analytics.loc[
    high_risk_mask & (df_analytics["isFraud"] == 1),
    "amount"
].sum()

value_capture_rate = (
    fraud_value_captured / fraud_value_total * 100
)

print(f"Total fraud value: ₹{fraud_value_total:,.2f}")
print(f"Fraud value captured: ₹{fraud_value_captured:,.2f}")
print(f"Fraud value capture rate: {value_capture_rate:.2f}%")

Total fraud value: ₹12,056,415,427.84
Fraud value captured: ₹8,373,717,346.95
Fraud value capture rate: 69.45%


In [32]:
ml_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_change",
    "destination_balance_change",
    "origin_depletion_ratio"
]

print(ml_features)

['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'origin_balance_change', 'destination_balance_change', 'origin_depletion_ratio']


In [33]:
# Sample normal transactions for anomaly model training
normal_sample = df_analytics[
    df_analytics["isFraud"] == 0
].sample(
    n=100000,
    random_state=42
)

X_train = normal_sample[ml_features].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

print(X_train.shape)

(100000, 8)


In [34]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

In [35]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

print(X_train_scaled.shape)

(100000, 8)


In [36]:
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_scaled)

print("Isolation Forest trained successfully.")

Isolation Forest trained successfully.


In [37]:
# Prepare ML features for the full dataset

X_full = df_analytics[ml_features].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

print(X_full.shape)

(6362620, 8)


In [38]:
X_full_scaled = scaler.transform(X_full)

print(X_full_scaled.shape)

(6362620, 8)


In [39]:
# Predict anomaly labels
# 1  = normal
# -1 = anomaly

anomaly_prediction = iso_forest.predict(X_full_scaled)

df_analytics["is_anomaly"] = (
    anomaly_prediction == -1
).astype(int)

print(
    df_analytics["is_anomaly"].value_counts()
)

is_anomaly
0    6296777
1      65843
Name: count, dtype: int64


In [40]:
# Isolation Forest decision function
# Lower values = more anomalous

raw_anomaly_score = iso_forest.decision_function(
    X_full_scaled
)

# Convert to a 0-100 anomaly risk score
anomaly_score_min = raw_anomaly_score.min()
anomaly_score_max = raw_anomaly_score.max()

df_analytics["anomaly_score"] = (
    100 *
    (anomaly_score_max - raw_anomaly_score)
    / (anomaly_score_max - anomaly_score_min)
)

df_analytics["anomaly_score"] = (
    df_analytics["anomaly_score"]
    .clip(0, 100)
)

df_analytics[
    [
        "transaction_id",
        "type",
        "amount",
        "is_anomaly",
        "anomaly_score",
        "isFraud"
    ]
].head(20)

,transaction_id,type,amount,is_anomaly,anomaly_score,isFraud
0,TXN_1,PAYMENT,9839.64,0,2.370210,0
1,TXN_2,PAYMENT,1864.28,0,0.957841,0
2,TXN_3,TRANSFER,181.00,0,0.186815,1
3,TXN_4,CASH_OUT,181.00,0,0.368948,1
4,TXN_5,PAYMENT,11668.14,0,1.123395,0
5,TXN_6,PAYMENT,7817.71,0,0.813598,0
6,TXN_7,PAYMENT,7107.77,0,2.604953,0
7,TXN_8,PAYMENT,7861.64,0,2.604953,0
8,TXN_9,PAYMENT,4024.36,0,0.349897,0
9,TXN_10,DEBIT,5337.77,0,1.150458,0


In [41]:
anomaly_evaluation = (
    df_analytics
    .groupby("is_anomaly")
    .agg(
        transactions=("transaction_id", "count"),
        fraud_transactions=("isFraud", "sum"),
        transaction_value=("amount", "sum"),
        fraud_value=("amount", lambda x:
            x[df_analytics.loc[x.index, "isFraud"] == 1].sum())
    )
    .reset_index()
)

anomaly_evaluation["fraud_rate_pct"] = (
    anomaly_evaluation["fraud_transactions"]
    / anomaly_evaluation["transactions"]
    * 100
)

anomaly_evaluation

,is_anomaly,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct
0,0,6296777,7384,9.924993e+11,6.677858e+09,0.117266
1,1,65843,829,1.518937e+11,5.378558e+09,1.259056


In [42]:
df_analytics.groupby("isFraud")["anomaly_score"].agg(
    ["count", "mean", "median", "min", "max"]
)

,count,mean,median,min,max
isFraud,,,,,
0,6354407,11.180913,4.371051,0.000000,100.000000
1,8213,30.032005,25.298254,0.069112,92.475262


In [43]:
import pandas as pd

# Assign additional risk points based on anomaly score
df_analytics["anomaly_risk_points"] = pd.cut(
    df_analytics["anomaly_score"],
    bins=[-1, 20, 50, 75, 100],
    labels=[0, 10, 20, 30],
    include_lowest=True
).astype(int)

# Combine corrected rule-based score with anomaly risk points
# Use risk_score_v2 for now; we will validate the corrected formula separately.
df_analytics["final_risk_score"] = (
    df_analytics["risk_score_v2"] +
    df_analytics["anomaly_risk_points"]
).clip(upper=100)

# Categorize final risk
df_analytics["final_risk_level"] = pd.cut(
    df_analytics["final_risk_score"],
    bins=[-1, 29, 49, 69, 100],
    labels=["Low", "Medium", "High", "Critical"]
)

print(df_analytics["final_risk_level"].value_counts())

final_risk_level
Medium      3227353
High        1691186
Low         1355210
Critical      88871
Name: count, dtype: int64


In [44]:
# CORRECTED FINAL RISK ENGINE
# Build the origin-balance indicator, corrected rule-based score, and final combined score.
# Destination balance mismatch is intentionally excluded from the corrected score.

df_analytics["origin_balance_anomaly"] = (
    df_analytics["origin_balance_mismatch"] == 0
).astype(int)

df_analytics["risk_score_corrected"] = (
    df_analytics["existing_flag"] * 35
    + df_analytics["zero_amount_flag"] * 35
    + df_analytics["origin_balance_anomaly"] * 25
    + df_analytics["full_origin_depletion"] * 20
    + df_analytics["type_risk_flag"] * 10
    + df_analytics["amount_above_p95"] * 10
).clip(upper=100)

df_analytics["anomaly_risk_points"] = pd.cut(
    df_analytics["anomaly_score"],
    bins=[-1, 20, 50, 75, 100],
    labels=[0, 10, 20, 30],
    include_lowest=True
).astype(int)

df_analytics["final_risk_score"] = (
    df_analytics["risk_score_corrected"]
    + df_analytics["anomaly_risk_points"]
).clip(upper=100)

df_analytics["final_risk_level"] = pd.cut(
    df_analytics["final_risk_score"],
    bins=[-1, 29, 49, 69, 100],
    labels=["Low", "Medium", "High", "Critical"]
)

print("Corrected rule-based and final risk scores created.")
print("\nFinal risk-level distribution:")
print(df_analytics["final_risk_level"].value_counts().reindex(
    ["Low", "Medium", "High", "Critical"], fill_value=0
))


Corrected rule-based and final risk scores created.

Final risk-level distribution:
final_risk_level
Low         4749847
Medium      1543126
High          63072
Critical       6575
Name: count, dtype: int64


In [45]:
total_fraud = df_analytics["isFraud"].sum()

high_critical = df_analytics["final_risk_level"].isin(
    ["High", "Critical"]
)

captured_fraud = df_analytics.loc[
    high_critical, "isFraud"
].sum()

total_fraud_value = df_analytics.loc[
    df_analytics["isFraud"] == 1, "amount"
].sum()

captured_fraud_value = df_analytics.loc[
    high_critical & (df_analytics["isFraud"] == 1), "amount"
].sum()

print(f"Total fraud transactions: {total_fraud:,}")
print(f"Fraud captured: {captured_fraud:,}")
print(f"Fraud capture rate: {captured_fraud / total_fraud * 100:.2f}%")
print(f"Fraud value capture rate: {captured_fraud_value / total_fraud_value * 100:.2f}%")

Total fraud transactions: 8,213
Fraud captured: 8,180
Fraud capture rate: 99.60%
Fraud value capture rate: 99.90%


In [46]:
# Compare fraud capture using the rule-based score alone
for score_col in ["risk_score", "risk_score_v2", "final_risk_score"]:
    level_col = {
        "risk_score": "risk_level",
        "risk_score_v2": "risk_level_v2",
        "final_risk_score": "final_risk_level"
    }[score_col]

    high_critical = df_analytics[level_col].isin(["High", "Critical"])
    captured = df_analytics.loc[high_critical, "isFraud"].sum()
    rate = captured / df_analytics["isFraud"].sum() * 100

    print(f"{score_col}: {captured:,} fraud transactions captured ({rate:.2f}%)")

risk_score: 16 fraud transactions captured (0.19%)
risk_score_v2: 2,618 fraud transactions captured (31.88%)
final_risk_score: 8,180 fraud transactions captured (99.60%)


In [47]:
risk_flags = [
    "existing_flag",
    "zero_amount_flag",
    "origin_balance_mismatch",
    "full_origin_depletion",
    "type_risk_flag",
    "amount_above_p95",
    "destination_balance_mismatch",
]

flag_summary = (
    df_analytics.groupby("isFraud")[risk_flags]
    .mean()
    .T
    .rename(columns={0: "normal_rate", 1: "fraud_rate"})
)

flag_summary["difference"] = (
    flag_summary["fraud_rate"] - flag_summary["normal_rate"]
)

display(flag_summary.sort_values("difference", ascending=False))

isFraud,normal_rate,fraud_rate,difference
full_origin_depletion,0.238035,0.975527,0.737492
type_risk_flag,0.434690,1.000000,0.565310
amount_above_p95,0.049635,0.332522,0.282886
existing_flag,0.000000,0.001948,0.001948
zero_amount_flag,0.000000,0.001948,0.001948
destination_balance_mismatch,0.601094,0.515768,-0.085327
origin_balance_mismatch,0.786972,0.005479,-0.781493


In [48]:
# FINAL RISK ENGINE — INITIAL COMBINED VERSION

#This stage combines **Version 2 rule-based risk** with Isolation Forest anomaly points.
#Its metrics are an intermediate result; the corrected score below is the final version used for export and downstream reporting.

In [49]:
final_risk_performance = (
    df_analytics
    .groupby("final_risk_level", observed=True)
    .agg(
        transactions=("transaction_id", "count"),
        fraud_transactions=("isFraud", "sum"),
        transaction_value=("amount", "sum"),
        fraud_value=("amount", lambda x:
            x[df_analytics.loc[x.index, "isFraud"] == 1].sum())
    )
    .reset_index()
)

final_risk_performance["fraud_rate_pct"] = (
    final_risk_performance["fraud_transactions"]
    / final_risk_performance["transactions"]
    * 100
)

final_risk_performance

,final_risk_level,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct
0,Low,4749847,25,5.490768e+11,9.535071e+06,0.000526
1,Medium,1543126,8,4.079508e+11,2.586875e+06,0.000518
2,High,63072,5448,1.455328e+11,2.922028e+09,8.637747
3,Critical,6575,2732,4.183255e+10,9.122265e+09,41.551331


In [50]:
# Fraud capture in High + Critical

high_critical = df_analytics["final_risk_level"].isin(
    ["High", "Critical"]
)

fraud_total = df_analytics["isFraud"].sum()

fraud_captured = df_analytics.loc[
    high_critical,
    "isFraud"
].sum()

fraud_value_total = df_analytics.loc[
    df_analytics["isFraud"] == 1,
    "amount"
].sum()

fraud_value_captured = df_analytics.loc[
    high_critical & (df_analytics["isFraud"] == 1),
    "amount"
].sum()

print(
    f"Fraud capture rate: "
    f"{fraud_captured / fraud_total * 100:.2f}%"
)

print(
    f"Fraud value capture rate: "
    f"{fraud_value_captured / fraud_value_total * 100:.2f}%"
)

Fraud capture rate: 99.60%
Fraud value capture rate: 99.90%


In [51]:
# Re-evaluate balance-related signals before finalizing the risk engine

balance_signal_check = (
    df_analytics.groupby(
        ["origin_balance_mismatch", "destination_balance_mismatch"],
        observed=True
    )
    .agg(
        transactions=("transaction_id", "count"),
        fraud_transactions=("isFraud", "sum"),
        transaction_value=("amount", "sum")
    )
    .reset_index()
)

balance_signal_check["fraud_rate_pct"] = (
    balance_signal_check["fraud_transactions"]
    / balance_signal_check["transactions"]
    * 100
)

balance_signal_check

,origin_balance_mismatch,destination_balance_mismatch,transactions,fraud_transactions,transaction_value,fraud_rate_pct
0,0,0,279916,3953,2.864839e+10,1.412209
1,0,1,1081920,4215,2.063359e+10,0.389585
2,1,0,2258870,24,7.339480e+11,0.001062
3,1,1,2741914,21,3.611630e+11,0.000766


In [52]:
# Create a semantically clear origin balance risk indicator

df_analytics["origin_balance_anomaly"] = (
    df_analytics["origin_balance_mismatch"] == 0
).astype(int)

# Destination balance mismatch is not strong enough to use as a
# primary risk signal, so we will not include it in the final score.

print(
    df_analytics.groupby("origin_balance_anomaly", observed=True)
    .agg(
        transactions=("transaction_id", "count"),
        fraud_transactions=("isFraud", "sum")
    )
    .assign(
        fraud_rate_pct=lambda x:
        x["fraud_transactions"] / x["transactions"] * 100
    )
)

                        transactions  fraud_transactions  fraud_rate_pct
origin_balance_anomaly                                                  
0                            5000784                  45        0.000900
1                            1361836                8168        0.599779


In [53]:
# CORRECTED RULE-BASED RISK SCORE
# This cell can be rerun independently after the feature-engineering cells.
# It uses origin_balance_anomaly and excludes destination_balance_mismatch.

df_analytics["origin_balance_anomaly"] = (
    df_analytics["origin_balance_mismatch"] == 0
).astype(int)

df_analytics["risk_score_corrected"] = (
    df_analytics["existing_flag"] * 35
    + df_analytics["zero_amount_flag"] * 35
    + df_analytics["origin_balance_anomaly"] * 25
    + df_analytics["full_origin_depletion"] * 20
    + df_analytics["type_risk_flag"] * 10
    + df_analytics["amount_above_p95"] * 10
).clip(upper=100)

print("Corrected rule-based score created.")
print(df_analytics["risk_score_corrected"].describe())


Corrected rule-based score created.
count    6.362620e+06
mean     1.498503e+01
std      1.287586e+01
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+01
75%      2.500000e+01
max      7.000000e+01
Name: risk_score_corrected, dtype: float64


In [54]:
# FINAL RISK ENGINE — CORRECTED VERSION
# Combine the corrected rule-based score with Isolation Forest anomaly points.

df_analytics["anomaly_risk_points"] = pd.cut(
    df_analytics["anomaly_score"],
    bins=[-1, 20, 50, 75, 100],
    labels=[0, 10, 20, 30],
    include_lowest=True
).astype(int)

df_analytics["final_risk_score"] = (
    df_analytics["risk_score_corrected"]
    + df_analytics["anomaly_risk_points"]
).clip(upper=100)

df_analytics["final_risk_level"] = pd.cut(
    df_analytics["final_risk_score"],
    bins=[-1, 29, 49, 69, 100],
    labels=["Low", "Medium", "High", "Critical"]
)

print("Final risk engine created.")
print(df_analytics["final_risk_level"].value_counts().reindex(
    ["Low", "Medium", "High", "Critical"], fill_value=0
))


Final risk engine created.
final_risk_level
Low         4749847
Medium      1543126
High          63072
Critical       6575
Name: count, dtype: int64


In [55]:
final_performance = (
    df_analytics.groupby("final_risk_level", observed=True)
    .agg(
        transactions=("transaction_id", "count"),
        fraud_transactions=("isFraud", "sum"),
        transaction_value=("amount", "sum"),
        fraud_value=(
            "amount",
            lambda x: x[df_analytics.loc[x.index, "isFraud"] == 1].sum()
        )
    )
    .reset_index()
)

final_performance["fraud_rate_pct"] = (
    final_performance["fraud_transactions"]
    / final_performance["transactions"]
    * 100
)

final_performance

,final_risk_level,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct
0,Low,4749847,25,5.490768e+11,9.535071e+06,0.000526
1,Medium,1543126,8,4.079508e+11,2.586875e+06,0.000518
2,High,63072,5448,1.455328e+11,2.922028e+09,8.637747
3,Critical,6575,2732,4.183255e+10,9.122265e+09,41.551331


In [56]:
# FINAL EVALUATION — FRAUD AND FRAUD-VALUE CAPTURE

#High/Critical capture is evaluated against the labelled fraud cases in this synthetic PaySim dataset.
#These are retrospective dataset metrics, not proof of production performance.

In [57]:
# Export the final risk-scored transaction dataset
output_columns = [
    "transaction_id", "step", "hour", "day", "type", "amount",
    "nameOrig", "oldbalanceOrg", "newbalanceOrig",
    "nameDest", "oldbalanceDest", "newbalanceDest",
    "isFraud", "isFlaggedFraud", "type_risk_flag",
    "zero_amount_flag", "full_origin_depletion",
    "origin_balance_anomaly", "amount_above_p95",
    "anomaly_score", "is_anomaly", "risk_score_corrected",
    "anomaly_risk_points", "final_risk_score", "final_risk_level"
]

missing_columns = [col for col in output_columns if col not in df_analytics.columns]
if missing_columns:
    raise KeyError(f"Missing columns required for export: {missing_columns}")

final_dataset = df_analytics[output_columns].copy()
print(f"Prepared export dataset: {len(final_dataset):,} rows, {len(final_dataset.columns)} columns")


Prepared export dataset: 6,362,620 rows, 25 columns


In [58]:
print(final_dataset.shape)
print(final_dataset.columns.tolist())
print(final_dataset["final_risk_level"].value_counts())

(6362620, 25)
['transaction_id', 'step', 'hour', 'day', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'type_risk_flag', 'zero_amount_flag', 'full_origin_depletion', 'origin_balance_anomaly', 'amount_above_p95', 'anomaly_score', 'is_anomaly', 'risk_score_corrected', 'anomaly_risk_points', 'final_risk_score', 'final_risk_level']
final_risk_level
Low         4749847
Medium      1543126
High          63072
Critical       6575
Name: count, dtype: int64


In [59]:
total_fraud = df_analytics["isFraud"].sum()

high_critical = df_analytics["final_risk_level"].isin(
    ["High", "Critical"]
)

captured_fraud = df_analytics.loc[
    high_critical, "isFraud"
].sum()

capture_rate = captured_fraud / total_fraud * 100

print(f"Total fraud transactions: {total_fraud:,}")
print(f"Fraud captured (High/Critical): {captured_fraud:,}")
print(f"Fraud capture rate: {capture_rate:.2f}%")

Total fraud transactions: 8,213
Fraud captured (High/Critical): 8,180
Fraud capture rate: 99.60%


In [60]:
from pathlib import Path

# Works when Jupyter is launched from the project root or the notebooks folder.
cwd = Path.cwd()
if (cwd / "data").exists():
    project_root = cwd
elif (cwd.parent / "data").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find the project data folder. Start Jupyter from the project root "
        "or the notebooks folder."
    )

output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_columns = [
    "transaction_id", "step", "hour", "day", "type", "amount",
    "nameOrig", "oldbalanceOrg", "newbalanceOrig",
    "nameDest", "oldbalanceDest", "newbalanceDest",
    "isFraud", "isFlaggedFraud", "type_risk_flag",
    "zero_amount_flag", "full_origin_depletion",
    "origin_balance_anomaly", "amount_above_p95",
    "anomaly_score", "is_anomaly", "risk_score_corrected",
    "anomaly_risk_points", "final_risk_score", "final_risk_level"
]

missing_columns = [col for col in output_columns if col not in df_analytics.columns]
if missing_columns:
    raise KeyError(f"Missing columns required for export: {missing_columns}")

final_dataset = df_analytics[output_columns].copy()
output_file = output_dir / "risk_scored_transactions.csv"
final_dataset.to_csv(output_file, index=False)

print("Export successful!")
print(f"Rows: {len(final_dataset):,}")
print(f"Columns: {len(final_dataset.columns)}")
print(f"Saved to: {output_file}")


Export successful!
Rows: 6,362,620
Columns: 25
Saved to: c:\Users\preet\OneDrive\Desktop\Financial Transaction Risk & Fraud Analytics Platform\outputs\risk_scored_transactions.csv
